In [1]:
#单元格1
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")

# --- 关键：确保项目根目录在路径中 ---
path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(path)

# --- 导入您的项目文件 ---
from utils.common_utils import printlog, set_seed, make_dir, calculate_and_save_metrics
from utils.common_params import *  #
from models.LD_Net import LD_Net

# --- 💡 关键修正：导入 CNNRNNs 类 ---
from models.CNNRNNs import CNNRNNs

from trainTest.early_stopping.early_stopping import EarlyStopping
from trainTest.optimizers.get_optimizer import Optimizer
from trainTest.losses.get_loss import get_classification_loss
from trainTest.lr_schedulers.get_lr_scheduler import LrScheduler
# 设置随机种子
set_seed(seed=42)
print("所有依赖导入成功。")

所有依赖导入成功。


In [2]:
# 单元格 2
# === 1. 数据集选择 ===
DATASET_NAME = "SIAT"
# --- 💡 关键修改 (方案 7-B) ---
DENOISE_METHOD = 'LD_Net_Online_Classification'
MODEL_TYPE = 'SC-BiGRU_Diff_Focal_7B'  # <-- 新的保存文件夹 (v2)
# --- 修改结束 ---
n_channels_check = 9

# === 2. 关键路径设置 ===
LD_NET_MODEL_PATH = os.path.join(
    os.getcwd(), 'checkpoints', 'LD_Net_SIAT', 'ld_net_best_model.pth'
)
base_data_path = os.path.join(path, "preProcessing")
DATA_DIR = os.path.join(
    base_data_path, "SIAT_LLMD_trainData", "Denoising_TrainSet_XY"
)

# --- 3. 模型与训练设置 ---
BATCH_SIZE = 64
LEARNING_RATE = 5e-4 # (0.0005) (这是分类器的 LR)
EPOCHS = 200
PATIENCE = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 4. 检查参数 ===
print(f"--- 方案 7-B: 特征工程 + Focal Loss (修复版) ---") # <-- 修改
print(f"设备: {DEVICE}")
print(f"将加载 LD-Net 降噪器: {os.path.abspath(LD_NET_MODEL_PATH)}")
print(f"将加载 *成对数据* (X, Y) 源: {os.path.abspath(DATA_DIR)}")
if C != n_channels_check:
    print(f"⚠️ 警告: common_params.py 中的 C={C} 与 SIAT (C=9) 不匹配!")
else:
    print("✅ C 参数检查通过。")

# === 5. 结果保存路径 ===
save_dir_root = os.path.join(path, 'results', DENOISE_METHOD, MODEL_TYPE)
save_dir_models_temp = save_dir_root
make_dir(save_dir_root)

print(f"✅ 最终结果 (CSV/PNG) 将保存到: {os.path.abspath(save_dir_root)}")
print(f"✅ 临时模型 (PTH) 将保存到: {os.path.abspath(save_dir_models_temp)}")

--- 方案 7-B: 特征工程 + Focal Loss (修复版) ---
设备: cuda
将加载 LD-Net 降噪器: E:\基于肌电图的下肢运动模式识别\Code2\trainTest\LD_train Test\checkpoints\LD_Net_SIAT\ld_net_best_model.pth
将加载 *成对数据* (X, Y) 源: E:\基于肌电图的下肢运动模式识别\Code2\preProcessing\SIAT_LLMD_trainData\Denoising_TrainSet_XY
✅ C 参数检查通过。
✅ 最终结果 (CSV/PNG) 将保存到: E:\基于肌电图的下肢运动模式识别\Code2\results\LD_Net_Online_Classification\SC-BiGRU_Diff_Focal_7B
✅ 临时模型 (PTH) 将保存到: E:\基于肌电图的下肢运动模式识别\Code2\results\LD_Net_Online_Classification\SC-BiGRU_Diff_Focal_7B


In [3]:
#单元格3
print("--- 正在加载 LD-Net 降噪模型 (步骤三的结果) ---")
try:
    model_denoise = torch.load(LD_NET_MODEL_PATH, weights_only=False)
    model_denoise.to(DEVICE)
    model_denoise.eval()
    print("✅ LD-Net 降噪模型加载成功并设为 eval() 模式。")
except Exception as e:
    print(f"❌ 加载 LD-Net 模型失败: {e}")
    print("请确保 步骤三 已成功运行，且 LD_NET_MODEL_PATH 路径正确。")
    raise e

--- 正在加载 LD-Net 降噪模型 (步骤三的结果) ---
✅ LD-Net 降噪模型加载成功并设为 eval() 模式。


In [4]:
#单元格4
import glob


class LdNetClassificationDataset(Dataset):
    """
    用于分类器训练的数据集。
    从 步骤一 的 'Denoising_TrainSet_XY' 文件夹加载数据。
    """

    def __init__(self, data_dir, subject_id):
        self.data_dir = data_dir
        self.subject_id = subject_id

        # --- 匹配 'Denoising_TrainSet_XY' 的文件名 ---
        file_name_prefix = '_XY_TrainData.npz'
        search_path = os.path.join(self.data_dir, f"{self.subject_id}{file_name_prefix}")
        file_paths = glob.glob(search_path)

        if not file_paths:
            print(f"警告: 在 {self.data_dir} 中未找到 {self.subject_id}{file_name_prefix}。")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))
            return

        try:
            data = np.load(file_paths[0])
            # 步骤一 保存的键是 'noisy_X' 和 'sub_motion_label_encoded'
            self.samples = data['noisy_X'].astype(np.float32)
            self.labels_encoded = data['sub_motion_label_encoded'].astype(np.int64)
            print(f"  已加载 {self.subject_id}: {self.samples.shape[0]} 个样本。")

        except Exception as e:
            print(f"加载 {file_paths[0]} 出错: {e}")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, idx):
        # 返回 噪声X 和 标签Y
        return self.samples[idx], self.labels_encoded[idx]


print("✅ 分类器数据集 (LdNetClassificationDataset) 定义完毕。")

✅ 分类器数据集 (LdNetClassificationDataset) 定义完毕。


In [5]:
# 单元格 5 (方案 7-B: 修复的特征工程)
from sklearn.model_selection import train_test_split
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.optim as optim
import numpy as np
import torch.nn as nn
from trainTest.metrics.metrics_utils import (
    get_accuracy, get_precision, get_recall, get_f1, get_specificity
)
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

# --- 💡 关键修改 (方案 7-B): 定义 Focal Loss (gamma=1.2) ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.2, reduction='mean'): # <-- gamma 设为 1.2
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        else: return focal_loss
# --- 修正结束 ---

# --- (本地指标函数 _calc_cls_metrics 保持不变) ---
def _calc_cls_metrics(y_true, y_pred, y_scores=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    metrics = {
        'Accuracy': get_accuracy(y_true, y_pred), 'Precision': get_precision(y_true, y_pred),
        'Recall': get_recall(y_true, y_pred), 'F1': get_f1(y_true, y_pred),
        'Specificity': get_specificity(y_true, y_pred) ,
    }
    auc_pct = 0.0
    if y_scores is not None:
        if isinstance(y_scores, list):
            try: y_scores = np.asarray(y_scores)
            except Exception: y_scores = None
        if isinstance(y_scores, np.ndarray) and y_scores.ndim == 2 and y_scores.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
                auc_pct = round(auc * 100.0, 3)
            except Exception: auc_pct = 0.0
    metrics['AUC'] = auc_pct
    return metrics
# --- 指标函数结束 ---

# --- (创建 2 通道输入 _create_2ch_input 保持不变) ---
def _create_2ch_input(out, C_channels=9):
    if isinstance(out, (list, tuple)): t = out[0]
    else: t = out
    if not (t.dim() == 3 and t.size(1) == C_channels):
         raise RuntimeError(f"LD-Net (eval) 输出形状 {tuple(t.shape)} 异常")
    ch1_signal = t.unsqueeze(1)
    ch2_diff = t[..., 1:] - t[..., :-1]
    ch2_diff_padded = F.pad(ch2_diff, (1, 0), 'constant', 0)
    ch2_diff_signal = ch2_diff_padded.unsqueeze(1)
    return torch.cat([ch1_signal, ch2_diff_signal], dim=1) # -> [B, 2, 9, L]

# --- (TwoChanStem 保持不变) ---
class TwoChanStem(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.fuse = nn.Conv2d(in_channels, out_channels, kernel_size=(1,1), stride=(1,1), padding=0, bias=True)
        self.bn   = nn.BatchNorm2d(out_channels)
        self.act  = nn.ReLU()
    def forward(self, x):
        x = self.fuse(x); x = self.bn(x); return self.act(x)

# --- (train_one_epoch 保持不变) ---
def train_one_epoch(model_classifier, twochan_stem,
                    model_denoiser, train_loader, criterion, optimizer, device):
    model_classifier.train(); twochan_stem.train()
    train_loss = 0.; total_num = 0; y_true_all = []; y_pred_all = []
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.no_grad():
            out = model_denoiser(inputs)
            x_2ch = _create_2ch_input(out, C_channels=C)
        inputs_denoised = twochan_stem(x_2ch)
        outputs = model_classifier(inputs_denoised)
        loss    = criterion(outputs, labels) # (现在这里是 Focal Loss gamma=1.2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        bs = labels.size(0)
        train_loss += loss.item() * bs; total_num  += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())
    return train_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, None)

# --- (test_one_epoch 保持不变) ---
@torch.no_grad()
def test_one_epoch(model_classifier, twochan_stem,
                   model_denoiser, test_loader, criterion, device):
    model_classifier.eval(); twochan_stem.eval(); model_denoiser.eval()
    test_loss, total_num = 0.0, 0
    y_true_all, y_pred_all, y_scores_all = [], [], []
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        out = model_denoiser(inputs)
        x_2ch = _create_2ch_input(out, C_channels=C)
        inputs_denoised = twochan_stem(x_2ch)
        outputs = model_classifier(inputs_denoised)
        loss    = criterion(outputs, labels) # (现在这里是 Focal Loss gamma=1.2)
        bs = labels.size(0)
        test_loss  += loss.item() * bs; total_num  += bs
        y_true_all.extend(labels.detach().cpu().numpy())
        y_pred_all.extend(torch.max(outputs, 1)[1].detach().cpu().numpy())
        y_scores_all.extend(F.softmax(outputs, dim=1).detach().cpu().numpy())
    return test_loss/total_num, _calc_cls_metrics(y_true_all, y_pred_all, np.asarray(y_scores_all)), y_true_all, y_pred_all

def train_test_subject(subject_id, data_dir, model_denoise, device, save_dir_models_temp,
                       model_type, epochs, batch_size, lr, patience, exp_tim):

    dataset = LdNetClassificationDataset(data_dir=data_dir, subject_id=subject_id)
    if len(dataset) == 0: return None
    train_idx, test_idx = train_test_split(
        list(range(len(dataset))), test_size=0.2, random_state=exp_tim, stratify=dataset.labels_encoded
    )

    # (删除静态类权重计算)

    train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(torch.utils.data.Subset(dataset, test_idx),  batch_size=batch_size, shuffle=False, num_workers=0)

    twochan_stem = TwoChanStem().to(device)
    model_classifier = CNNRNNs(rnn_type='BiGRU').to(device)
    print("已实例化 1x1 Stem 适配器 和 SC-BiGRU 分类器。")

    # --- 💡 关键修改 (方案 7-B): 使用 Focal Loss (gamma=1.2) ---
    criterion = FocalLoss(gamma=1.2).to(device) # (使用 gamma=1.2)
    print("  > 使用 FocalLoss (gamma=1.2) 作为损失函数。")
    # --- 修正结束 ---

    # --- 💡 关键修改 (方案 7-B): 修复优化器 (判别性 LR) ---
    print("正在为 Stem 和 分类器 设置优化器 (LD-Net 已冻结)...")

    stem_lr = lr * 2.0 # (例如 5e-4 * 2 = 1e-3，Stem 从头学，需要高 LR)
    classifier_lr = lr # (例如 5e-4，BiGRU 已经很成熟)

    print(f"  > 使用 torch.optim.AdamW (判别性 LR): Stem={stem_lr:.0e}, Classifier={classifier_lr:.0e}")

    # (我们绕过 Optimizer 类，直接使用 AdamW)
    optimizer = optim.AdamW([
        {'params': twochan_stem.parameters(), 'lr': stem_lr},
        {'params': model_classifier.parameters(), 'lr': classifier_lr}
    ], lr=lr, weight_decay=1e-4) # (lr=lr 是 AdamW 的默认值，实际生效的是组LR)
    # --- 修正结束 ---

    # 4. Scheduler & EarlyStopping (不变)
    scheduler_params = {
        'ReduceLROnPlateau': {
            'mode':'min','factor':0.1,'patience':5,'threshold':1e-4,'min_lr':1e-8
        }
    }
    # (注意：Scheduler 现在会同时降低 stem_lr 和 classifier_lr)
    scheduler = LrScheduler(optimizer=optimizer, scheduler_type='ReduceLROnPlateau',
                            params=scheduler_params,
                            max_epoch=epochs).get_scheduler()

    save_dir_subj = os.path.join(save_dir_models_temp, subject_id)
    make_dir(save_dir_subj)
    best_loss_path = os.path.join(save_dir_subj, f'best_loss_model_exp{exp_tim}.pth')
    early_stopping  = EarlyStopping(patience=patience, verbose=True, path=best_loss_path)
    best_acc = -1.0
    best_acc_path = os.path.join(save_dir_subj, f'best_acc_model_exp{exp_tim}.pth')

    def save_models_checkpoint(path):
        torch.save({
            'twochan_stem_state_dict': twochan_stem.state_dict(),
            'model_classifier_state_dict': model_classifier.state_dict(),
        }, path)

    # 5. 训练循环
    for epoch in range(1, epochs+1):
        train_loss, train_metrics = train_one_epoch(
            model_classifier, twochan_stem, model_denoise,
            train_loader, criterion, optimizer, device
        )
        val_loss, val_metrics, _, _ = test_one_epoch(
            model_classifier, twochan_stem, model_denoise,
            test_loader, criterion, device
        )
        scheduler.step(val_loss)
        model_state_dict_to_save = {
            'twochan_stem_state_dict': twochan_stem.state_dict(),
            'model_classifier_state_dict': model_classifier.state_dict(),
        }
        early_stopping(val_loss, model_state_dict_to_save)
        val_acc = val_metrics['Accuracy']
        if val_acc > best_acc:
            best_acc = val_acc
            save_models_checkpoint(best_acc_path)
            print(f"    {subject_id} | Exp {exp_tim} | 新的最佳准确率: {val_acc:.2f}% (已保存)")
        if (epoch % 10 == 0) or early_stopping.early_stop:
            print(f"{subject_id} | Exp {exp_tim} | Ep {epoch:03d} | "
                  f"Train {train_loss:.4f}/{train_metrics['Accuracy']:.2f}% | "
                  f"Val {val_loss:.4f}/{val_metrics['Accuracy']:.2f}%")
        if early_stopping.early_stop:
            print("早停触发 (基于 Val Loss)。"); break

    # 6. 加载最佳
    load_path = best_acc_path if os.path.exists(best_acc_path) else best_loss_path
    print(f"加载 {subject_id} (Exp {exp_tim}) 的最佳模型 (来自: {load_path})...")
    best_stem = TwoChanStem().to(device)
    best_classifier = CNNRNNs(rnn_type='BiGRU').to(device)
    ckpt = torch.load(load_path)
    best_stem.load_state_dict(ckpt['twochan_stem_state_dict'])
    best_classifier.load_state_dict(ckpt['model_classifier_state_dict'])

    # 7. 最终评估
    test_loss_final, test_metrics_final, y_true_final, y_pred_final = test_one_epoch(
        best_classifier, best_stem, model_denoise,
        test_loader, criterion, device
    )

    # 8. 混淆矩阵 (不变)
    print('正在计算并保存混淆矩阵...')
    motions_list_cm = ['WAK', 'STDUP', 'SITDN', 'UPS', 'DNS']
    cm_save_jpg_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.jpg')
    cm_save_csv_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.xlsx')
    plot_confusion_matrix = PlotConfusionMatrix(
        y_true_final, y_pred_final, label_type=motions_list_cm,
        show_type='all', plot=False, save_fig=True, save_results=True, cmap='YlGnBu'
    )
    plot_confusion_matrix.get_confusion_matrix(cm_save_jpg_name, cm_save_csv_name)
    print(f'混淆矩阵已保存到: {save_dir_subj}')

    test_metrics_final['Subject'] = subject_id
    return test_metrics_final

print("✅ 训练/测试函数 (train_test_subject) [方案7-B: 修复优化器+Focal(1.2)] 定义完毕。")

✅ 训练/测试函数 (train_test_subject) [方案7-B: 修复优化器+Focal(1.2)] 定义完毕。


In [6]:
#单元格6
import pandas as pd
import numpy as np

# (来自 common_params.py)
subjects_list = ['Sub01', 'Sub02', 'Sub03', 'Sub04', 'Sub05', 'Sub31', 'Sub32', 'Sub33', 'Sub34', 'Sub35']

# 1. --- 初始化列表 ---
metrics_all_subjects = []

# (确保 单元格 2 已运行)
printlog(f"--- 开始在 {DATASET_NAME} 上执行 {DENOISE_METHOD} + {MODEL_TYPE} 受试者内测试 ---", time=True,
         line_break=True)

# 2. --- 主训练循环 (与你之前的一样) ---
for subject in subjects_list:
    for exp_tim in range(1, K_of_repeated_experiments + 1):

        printlog(f"--- 正在处理受试者: {subject} | 实验: {exp_tim}/{K_of_repeated_experiments} ---", time=True, line_break=True)

        test_metrics = train_test_subject(
            subject_id=subject,
            data_dir=DATA_DIR,
            model_denoise=model_denoise,
            device=DEVICE,
            save_dir_models_temp=save_dir_models_temp,
            model_type=MODEL_TYPE,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            patience=PATIENCE,
            exp_tim=exp_tim
        )

        if test_metrics:
            test_metrics['Experiment'] = exp_tim
            metrics_all_subjects.append(test_metrics)
            print(f"--- 受试者 {subject} 实验 {exp_tim} 完成。最终测试 Acc: {test_metrics['Accuracy']:.2f}% ---")
# --- 主循环结束 ---


printlog(f"--- 所有受试者处理完毕 ---", time=True, line_break=True)

# 3. --- 💡 关键修改：计算并保存为【图片样式】 ---
if metrics_all_subjects:
    # 3.1. 保存【所有50次运行】的详细原始数据
    df_summary = pd.DataFrame(metrics_all_subjects)
    save_path_summary_by_subject = os.path.join(save_dir_root, 'summary_by_subject.csv')
    df_summary.to_csv(save_path_summary_by_subject, index=False, float_format='%.3f')
    print(f"✅ 包含50次运行的详细CSV已保存到: {save_path_summary_by_subject}")

    # 3.2. 计算【每个受试者的平均值】(数字)
    df_avg_by_subject = df_summary.groupby('Subject').mean(numeric_only=True)
    # 计算【每个受试者的标准差】(数字)
    df_std_by_subject = df_summary.groupby('Subject').std(numeric_only=True)

    # 3.3. 创建【受试者10行】的格式化 (mean+std) DataFrame
    df_formatted_subjects = pd.DataFrame(index=df_avg_by_subject.index)

    # 确定要格式化的列 (排除 'Experiment')
    metrics_cols = [col for col in df_avg_by_subject.columns if col != 'Experiment']

    for col in metrics_cols:
        # 格式化为 "均值 + 标准差"，并保留4位小数 (根据图片)
        df_formatted_subjects[col] = df_avg_by_subject[col].map('{:.4f}'.format) + \
                                     "+" + \
                                     df_std_by_subject[col].map('{:.4f}'.format)

    # 3.4. 计算【所有受试者平均值的平均值】(数字)
    df_total_avg = df_avg_by_subject[metrics_cols].mean().to_frame().T

    # 3.5. 计算【所有受试者平均值的标准差】(数字)
    df_total_std = df_avg_by_subject[metrics_cols].std().to_frame().T

    # 3.6. 将总平均值和总标准差格式化为DataFrame (用于拼接)
    # 只保留小数 (根据图片)
    df_total_avg_formatted = df_total_avg.applymap(lambda x: f"{x:.4f}")
    df_total_std_formatted = df_total_std.applymap(lambda x: f"{x:.4f}")

    # 更改索引名以便拼接
    df_total_avg_formatted.index = ['mean']
    df_total_std_formatted.index = ['std']

    # 3.7. 拼接所有部分：10行 (mean+std) + 1行 (total mean) + 1行 (total std)
    df_final_report = pd.concat([df_formatted_subjects, df_total_avg_formatted, df_total_std_formatted])

    # 3.8. 保存最终的报告
    save_path_final_report = os.path.join(save_dir_root, 'all_metrics_averaged_results.csv')
    # index=True 是因为 Subject/mean/std 现在是索引
    df_final_report.to_csv(save_path_final_report, index=True)
    print(f"✅ 【最终报告 (mean+std)】(12行) 已保存到: {save_path_final_report}")

    # (现在 "alone_subject_averaged_results.csv" 已经合并到上面了，不再单独保存)

    # 3.9. 检查 Sub02 (使用未格式化的 df_avg_by_subject)
    try:
        sub02_new_avg = df_avg_by_subject.loc['Sub02']['Accuracy']
        print(f"\n--- 检查 Sub02 ---")
        print(f"Sub02 新的5次实验平均准确率: {sub02_new_avg:.3f}%")
    except Exception as e:
        print(f"无法自动检查 Sub02 的新平均值: {e}")

else:
    print("没有受试者被成功处理。")


============================================================2025-11-11 18:51:28
--- 开始在 SIAT 上执行 LD_Net_Online_Classification + SC-BiGRU_Diff_Focal_7B 受试者内测试 ---...


============================================================2025-11-11 18:51:28
--- 正在处理受试者: Sub01 | 实验: 1/5 ---...

  已加载 Sub01: 690 个样本。
已实例化 1x1 Stem 适配器 和 SC-BiGRU 分类器。
  > 使用 FocalLoss (gamma=1.2) 作为损失函数。
正在为 Stem 和 分类器 设置优化器 (LD-Net 已冻结)...
  > 使用 torch.optim.AdamW (判别性 LR): Stem=1e-03, Classifier=5e-04
Validation loss decreased (inf --> 1.229510).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 27.54% (已保存)
Validation loss decreased (1.229510 --> 1.222869).  Saving model ...
Validation loss decreased (1.222869 --> 1.197887).  Saving model ...
Validation loss decreased (1.197887 --> 1.124292).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 50.73% (已保存)
Validation loss decreased (1.124292 --> 0.979044).  Saving model ...
    Sub01 | Exp 1 | 新的最佳准确率: 82.61% (已保存)
Validation loss decreased (0.979044 --> 0.739466).  Sav